# Сколько стоит опыт

Третий вопрос: что даёт уровень должности и как с ним связана удалённая работа.

Уровень выводится из названия вакансии — hh.ru отдельного поля для него не
держит. Там, где уровень не назван, вакансия остаётся в «не указан», и её
медиана не считается: «не указан» — это не уровень.

In [ ]:
import sys
from pathlib import Path


def find_root() -> Path:
    """Корень репозитория: ищем pyproject.toml вверх по дереву."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return here


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from vacancy_dashboard import analysis, clean, collect  # noqa: E402

RAW = ROOT / "data" / "raw" / "vacancies.json"
FIXTURE = ROOT / "tests" / "fixtures" / "vacancies_sample.json"

source = RAW if RAW.exists() else FIXTURE
print(f"Источник данных: {source}")
if source == FIXTURE:
    print(
        "ВНИМАНИЕ: это синтетическая фикстура, а не рыночные данные. "
        "Числа отсюда нельзя публиковать."
    )

frame = clean.drop_archived(clean.build_frame(collect.load(source)))
print(f"Вакансий в выборке: {len(frame)}")
frame.head()

In [ ]:
result, figure = analysis.question_experience(frame)
print(result.text)
result.table

In [ ]:
figure

## Доля удалённой работы по уровням

Считается по всем вакансиям уровня, в том числе без зарплаты.

In [ ]:
grouped = frame.groupby("grade")["work_format"].value_counts(normalize=True).unstack(fill_value=0)
grouped